# VALIDACION CON DATOS DE LABORATORIO

In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print("====================================================================")
print("     MÓDULO DE AUDITORÍA CONTINUA: VALIDACIÓN CONTRA LABORATORIO   ")
print("====================================================================\n")

# 1. Carga de Predicciones Continuas del Soft-Sensor (Alta Frecuencia)
df_predictions = pd.read_csv('data/final/soft_sensor_continuous_predictions.csv')
df_predictions['timestamp'] = pd.to_datetime(df_predictions['timestamp'])
df_predictions = df_predictions.sort_values('timestamp')

# 2. Carga de Ensayos Reales de Laboratorio Limpios (Baja Frecuencia)
df_lab = pd.read_csv('data/processed/assays_clean_no_duplicates.csv')
df_lab['timestamp'] = pd.to_datetime(df_lab['timestamp'])
df_lab = df_lab.sort_values('timestamp')

print(f"[DataOps] Registros continuos del Soft-Sensor: {len(df_predictions)}")
print(f"[DataOps] Ensayos analíticos reales de Lab:    {len(df_lab)}")

# 3. Alineamiento Temporal Dinámico (merge_asof)
# Buscamos la predicción del soft-sensor más cercana en el pasado para cada dato de laboratorio
# tolerance=pd.Timedelta('30min') asegura que no crucemos datos con desfases absurdos
df_audit = pd.merge_asof(
    df_lab, 
    df_predictions, 
    on='timestamp', 
    direction='backward', 
    tolerance=pd.Timedelta('30min')
)

# Eliminar filas donde no hubo coincidencia temporal dentro de la tolerancia
df_audit = df_audit.dropna(subset=['pCu_Est', 'pFe_Est', 'pMo_Est']).reset_index(drop=True)
print(f"[Match] Puntos de control síncronos encontrados para auditoría: {len(df_audit)}\n")

# 4. Mapeo de pares analíticos (Verdad Terreno vs Estimación)
elements_mapping = {
    'Cobre (pCu)': {'real': 'pCu', 'pred': 'pCu_Est'},
    'Hierro (pFe)': {'real': 'pFe', 'pred': 'pFe_Est'},
    'Molibdeno (pMo)': {'real': 'pMo', 'pred': 'pMo_Est'}
}

# 5. Cálculo y Despliegue de Métricas Globales de Producción
print("=== CUADRO ESTADÍSTICO DE VALIDACIÓN DEL SOFT-SENSOR CONTINUO ===")
print(f"{'Elemento Mineral':<18} | {'RMSE':<8} | {'MAE':<8} | {'R² Global':<8}")
print("-" * 55)

for name, keys in elements_mapping.items():
    y_real = df_audit[keys['real']].values
    y_pred = df_audit[keys['pred']].values
    
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    mae = mean_absolute_error(y_real, y_pred)
    r2 = r2_score(y_real, y_pred)
    
    print(f"{name:<18} | {rmse:.4f} | {mae:.4f} | {r2:.4f}")

# 6. Exportar matriz de auditoría para generar gráficos en la Tesis (Scatter plots / Error histórico)
df_audit.to_csv('data/final/soft_sensor_audit_results.csv', index=False)
print("\n[INFO] Matriz unificada de auditoría exportada en: 'soft_sensor_audit_results.csv'")
print("Recomendación: Usa este archivo para graficar Ley Real vs Ley Estimada a lo largo del tiempo.")

     MÓDULO DE AUDITORÍA CONTINUA: VALIDACIÓN CONTRA LABORATORIO   

[DataOps] Registros continuos del Soft-Sensor: 18124
[DataOps] Ensayos analíticos reales de Lab:    611
[Match] Puntos de control síncronos encontrados para auditoría: 533

=== CUADRO ESTADÍSTICO DE VALIDACIÓN DEL SOFT-SENSOR CONTINUO ===
Elemento Mineral   | RMSE     | MAE      | R² Global
-------------------------------------------------------
Cobre (pCu)        | 2.4554 | 1.9790 | -0.8324
Hierro (pFe)       | 1.4903 | 1.1761 | -0.4203
Molibdeno (pMo)    | 0.8655 | 0.7421 | -0.6826

[INFO] Matriz unificada de auditoría exportada en: 'soft_sensor_audit_results.csv'
Recomendación: Usa este archivo para graficar Ley Real vs Ley Estimada a lo largo del tiempo.
